In [ ]:
import pandas as pd

test_df = pd.read_csv('/content/test_data.csv')
train_df = pd.read_csv('/content/train_data.csv')
title_brand_df = pd.read_csv('/content/title_brand.csv')

print('Test Data Head:')
display(test_df.head())

print('\nTrain Data Head:')
display(train_df.head())

print('\nTitle Brand Data Head:')
display(title_brand_df.head())

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from wordcloud import WordCloud, STOPWORDS
import warnings
warnings.filterwarnings('ignore')

# خواندن داده‌ها
train_df = pd.read_csv('/content/train_data.csv', low_memory=False)
test_df = pd.read_csv('/content/test_data.csv')
title_brand_df = pd.read_csv('/content/title_brand.csv')

print(f"Train shape: {train_df.shape}")
print(f"Test shape: {test_df.shape}")
print(f"Title-Brand shape: {title_brand_df.shape}")

In [ ]:
# رسم توزیع overall
plt.figure(figsize=(10, 6))
overall_counts = train_df['overall'].value_counts().sort_index()
colors = ['#e74c3c', '#e67e22', '#f39c12', '#27ae60', '#2ecc71']
bars = plt.bar(overall_counts.index, overall_counts.values, color=colors, edgecolor='black')

# اضافه کردن تعداد روی هر ستون
for bar, count in zip(bars, overall_counts.values):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1000,
             f'{count:,}', ha='center', va='bottom', fontsize=10)

plt.xlabel('امتیاز (Overall)', fontsize=12)
plt.ylabel('تعداد نظرات', fontsize=12)
plt.title('توزیع امتیازات نظرات', fontsize=14)
plt.xticks([1, 2, 3, 4, 5])
plt.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

# نمایش درصدها
print("\n📊 توزیع امتیازات:")
print("-" * 40)
for rating in sorted(train_df['overall'].unique()):
    count = (train_df['overall'] == rating).sum()
    percentage = count / len(train_df) * 100
    print(f"امتیاز {rating}: {count:,} نظر ({percentage:.2f}%)")

# بررسی عدم توازن
print("\n" + "=" * 50)
print("🔍 تحلیل توازن داده‌ها:")
print("=" * 50)
imbalance_ratio = train_df['overall'].value_counts().max() / train_df['overall'].value_counts().min()
print(f"نسبت عدم توازن (بیشترین/کمترین): {imbalance_ratio:.2f}")

In [ ]:
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
import re

# دانلود منابع NLTK
nltk.download('stopwords', quiet=True)
nltk.download('punkt', quiet=True)

# تعریف دسته‌بندی احساسات
def categorize_sentiment(overall):
    if overall >= 4:
        return 'positive'
    elif overall == 3:
        return 'neutral'
    else:
        return 'negative'

train_df['sentiment'] = train_df['overall'].apply(categorize_sentiment)

# تابع پاکسازی متن
def clean_text(text):
    if pd.isna(text):
        return ""
    text = str(text).lower()
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

# Stop words سفارشی
custom_stopwords = set(STOPWORDS)
custom_stopwords.update(stopwords.words('english'))
# اضافه کردن کلمات عمومی که اطلاعات خاصی ندارند
custom_stopwords.update(['one', 'two', 'would', 'could', 'also', 'get', 'got',
                          'use', 'used', 'using', 'thing', 'things', 'really',
                          'even', 'much', 'well', 'still', 'just', 'like',
                          'amazon', 'product', 'item', 'buy', 'bought', 'purchase',
                          'ordered', 'order', 'received', 'came', 'arrived'])

# ایجاد Word Cloud برای هر دسته
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

sentiments = ['positive', 'neutral', 'negative']
titles = ['😊 احساس مثبت (امتیاز ۴-۵)', '😐 احساس خنثی (امتیاز ۳)', '😞 احساس منفی (امتیاز ۱-۲)']
colors = ['Greens', 'Blues', 'Reds']

for idx, (sentiment, title, colormap) in enumerate(zip(sentiments, titles, colors)):
    # فیلتر کردن نظرات
    texts = train_df[train_df['sentiment'] == sentiment]['reviewText'].dropna()

    # ترکیب و پاکسازی متن‌ها (نمونه‌گیری برای سرعت بیشتر)
    sample_size = min(50000, len(texts))
    combined_text = ' '.join(texts.sample(sample_size, random_state=42).apply(clean_text))

    # ایجاد Word Cloud
    wordcloud = WordCloud(
        width=800,
        height=400,
        background_color='white',
        stopwords=custom_stopwords,
        max_words=100,
        colormap=colormap,
        min_font_size=10,
        max_font_size=100
    ).generate(combined_text)

    axes[idx].imshow(wordcloud, interpolation='bilinear')
    axes[idx].set_title(title, fontsize=14, fontweight='bold')
    axes[idx].axis('off')

plt.suptitle('ابر کلمات نظرات بر اساس احساس', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# تحلیل کلمات مشترک
from collections import Counter

def get_top_words(texts, n=50):
    all_words = []
    for text in texts.dropna().sample(min(30000, len(texts)), random_state=42):
        cleaned = clean_text(text)
        words = [w for w in cleaned.split() if w not in custom_stopwords and len(w) > 2]
        all_words.extend(words)
    return Counter(all_words).most_common(n)

positive_words = set([w[0] for w in get_top_words(train_df[train_df['sentiment'] == 'positive']['reviewText'])])
negative_words = set([w[0] for w in get_top_words(train_df[train_df['sentiment'] == 'negative']['reviewText'])])
common_words = positive_words.intersection(negative_words)

print("📝 تحلیل کلمات مشترک بین نظرات مثبت و منفی:")
print("=" * 50)
print(f"\n🔄 کلمات مشترک ({len(common_words)} کلمه):")
print(list(common_words)[:20])

print("""
\n💡 تفسیر:
-----------
کلمات مشترک معمولاً شامل موارد زیر هستند:
۱. نام محصولات و برندها (cable, battery, phone, etc.)
۲. کلمات توصیفی عمومی (quality, price, time)
۳. کلمات مرتبط با عملکرد (work, works, working)

این کلمات در هر دو دسته ظاهر می‌شوند چون:
- در نظرات مثبت: "great quality", "works perfectly"
- در نظرات منفی: "poor quality", "doesn't work"

بنابراین context و کلمات همراه مهم‌تر از خود کلمه هستند.
""")

In [ ]:
# محاسبه مجموع vote برای هر نظردهنده
train_df['vote'] = pd.to_numeric(train_df['vote'], errors='coerce').fillna(0)

reviewer_votes = train_df.groupby(['reviewerID', 'reviewerName']).agg({
    'vote': 'sum',
    'reviewText': 'count'
}).reset_index()
reviewer_votes.columns = ['reviewerID', 'reviewerName', 'total_votes', 'review_count']

# مرتب‌سازی و انتخاب ۱۰ نفر برتر
top_10_reviewers = reviewer_votes.nlargest(10, 'total_votes')

print("🏆 ۱۰ نظردهنده برتر بر اساس مجموع vote:")
print("=" * 70)
print(f"{'رتبه':<6}{'نام':<30}{'مجموع Vote':<15}{'تعداد نظرات':<15}")
print("-" * 70)

for idx, row in enumerate(top_10_reviewers.itertuples(), 1):
    name = str(row.reviewerName)[:28] if pd.notna(row.reviewerName) else 'Unknown'
    print(f"{idx:<6}{name:<30}{int(row.total_votes):<15,}{row.review_count:<15}")

# نمایش به صورت جدول
print("\n📊 جدول خلاصه:")
display_df = top_10_reviewers[['reviewerName', 'total_votes', 'review_count']].copy()
display_df.columns = ['نام نظردهنده', 'مجموع Vote', 'تعداد نظرات']
display_df.index = range(1, 11)
display_df

In [ ]:
# محاسبه طول متن
train_df['text_length'] = train_df['reviewText'].astype(str).apply(len)

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# هیستوگرام اصلی
axes[0].hist(train_df['text_length'], bins=100, color='steelblue', edgecolor='black', alpha=0.7)
axes[0].set_xlabel('طول متن (تعداد کاراکتر)', fontsize=12)
axes[0].set_ylabel('تعداد نظرات', fontsize=12)
axes[0].set_title('هیستوگرام طول متن - حالت اصلی', fontsize=14)
axes[0].axvline(train_df['text_length'].median(), color='red', linestyle='--', label=f'میانه: {train_df["text_length"].median():.0f}')
axes[0].legend()

# هیستوگرام فیلترشده
# حذف outliers با استفاده از percentile
lower_bound = train_df['text_length'].quantile(0.01)
upper_bound = train_df['text_length'].quantile(0.99)
filtered_df = train_df[(train_df['text_length'] >= lower_bound) & (train_df['text_length'] <= upper_bound)]

axes[1].hist(filtered_df['text_length'], bins=50, color='forestgreen', edgecolor='black', alpha=0.7)
axes[1].set_xlabel('طول متن (تعداد کاراکتر)', fontsize=12)
axes[1].set_ylabel('تعداد نظرات', fontsize=12)
axes[1].set_title(f'هیستوگرام طول متن - فیلترشده (صدک ۱ تا ۹۹)', fontsize=14)
axes[1].axvline(filtered_df['text_length'].median(), color='red', linestyle='--', label=f'میانه: {filtered_df["text_length"].median():.0f}')
axes[1].legend()

plt.tight_layout()
plt.show()

# آمار توصیفی
print("\n📊 آمار توصیفی طول متن:")
print("=" * 50)
print(f"حداقل: {train_df['text_length'].min():,}")
print(f"حداکثر: {train_df['text_length'].max():,}")
print(f"میانگین: {train_df['text_length'].mean():,.2f}")
print(f"میانه: {train_df['text_length'].median():,.0f}")
print(f"انحراف معیار: {train_df['text_length'].std():,.2f}")
print(f"\nصدک‌ها:")
for p in [25, 50, 75, 90, 95, 99]:
    print(f"  صدک {p}: {train_df['text_length'].quantile(p/100):,.0f}")

In [ ]:
print("""
💡 پیشنهاد برای محدودیت طول متن در مدل‌سازی:
=============================================
بر اساس توزیع داده‌ها:
- حداقل پیشنهادی: 10 کاراکتر (حذف نظرات خیلی کوتاه و بی‌معنی)
- حداکثر پیشنهادی: 2000-3000 کاراکتر (صدک 95-99)

دلایل:
۱. نظرات خیلی کوتاه معمولاً اطلاعات کافی ندارند
۲. نظرات خیلی بلند ممکن است شامل اطلاعات اضافی باشند و حافظه زیادی مصرف کنند
۳. برای مدل‌های Transformer معمولاً محدودیت 512 توکن وجود دارد

🎯 بازه پیشنهادی: [10, 2500] کاراکتر
""")

In [ ]:
# فیلتر نظرات با امتیاز ۵
five_star_reviews = train_df[train_df['overall'] == 5]

# شمارش تعداد امتیاز ۵ برای هر محصول
product_five_stars = five_star_reviews.groupby('asin').size().reset_index(name='five_star_count')

# مرتب‌سازی و انتخاب ۱۰ محصول برتر
top_10_products = product_five_stars.nlargest(10, 'five_star_count')

# ادغام با اطلاعات برند و عنوان
top_10_products = top_10_products.merge(title_brand_df, on='asin', how='left')

# نمایش نتایج
print("🌟 ۱۰ محصول برتر با بیشترین امتیاز ۵:")
print("=" * 100)

result_df = top_10_products[['brand', 'title', 'five_star_count']].copy()
result_df.columns = ['برند', 'عنوان محصول', 'تعداد امتیاز ۵']
result_df.index = range(1, 11)

# نمایش با محدودیت طول عنوان
for idx, row in result_df.iterrows():
    brand = str(row['برند'])[:25] if pd.notna(row['برند']) else 'نامشخص'
    title = str(row['عنوان محصول'])[:50] + '...' if pd.notna(row['عنوان محصول']) and len(str(row['عنوان محصول'])) > 50 else str(row['عنوان محصول'])
    print(f"{idx}. برند: {brand}")
    print(f"   عنوان: {title}")
    print(f"   تعداد امتیاز ۵: {row['تعداد امتیاز ۵']:,}")
    print("-" * 80)

In [ ]:
# ادغام داده‌های آموزش با اطلاعات برند
train_with_brand = train_df.merge(title_brand_df[['asin', 'brand']], on='asin', how='left')

# شمارش تعداد نظرات برای هر برند
brand_review_counts = train_with_brand.groupby('brand').size().reset_index(name='review_count')

# انتخاب ۱۰ برند با بیشترین نظر
top_10_brands = brand_review_counts.nlargest(10, 'review_count')['brand'].tolist()

# محاسبه میانگین امتیاز برای این ۱۰ برند
brand_stats = train_with_brand[train_with_brand['brand'].isin(top_10_brands)].groupby('brand').agg({
    'overall': 'mean',
    'reviewText': 'count'
}).reset_index()
brand_stats.columns = ['برند', 'میانگین امتیاز', 'تعداد نظرات']

# مرتب‌سازی بر اساس میانگین امتیاز
brand_stats = brand_stats.sort_values('میانگین امتیاز', ascending=False)
brand_stats.index = range(1, 11)

print("🏢 ۱۰ برند با بیشترین تعداد نظر (مرتب بر اساس میانگین امتیاز):")
print("=" * 60)
print(brand_stats.to_string())

# نمودار
fig, ax = plt.subplots(figsize=(12, 6))
bars = ax.barh(brand_stats['برند'], brand_stats['میانگین امتیاز'], color='steelblue', edgecolor='black')
ax.set_xlabel('میانگین امتیاز', fontsize=12)
ax.set_ylabel('برند', fontsize=12)
ax.set_title('میانگین امتیاز ۱۰ برند پرنظر', fontsize=14)
ax.set_xlim(0, 5)

# اضافه کردن مقادیر
for bar, val in zip(bars, brand_stats['میانگین امتیاز']):
    ax.text(val + 0.05, bar.get_y() + bar.get_height()/2, f'{val:.2f}', va='center')

plt.tight_layout()
plt.show()

In [ ]:
# نصب کتابخانه‌های مورد نیاز
!pip install gensim -q

import gensim.downloader as api
from gensim.models import KeyedVectors

# بارگذاری مدل word2vec پیش‌آموخته
print("در حال بارگذاری مدل Word2Vec...")
word2vec_model = api.load('word2vec-google-news-300')
print("مدل با موفقیت بارگذاری شد!")

In [ ]:
# پیدا کردن کلمات مشابه warranty و guarantee
def get_similar_words(model, words, topn=20):
    similar_words = set()
    for word in words:
        try:
            similars = model.most_similar(word, topn=topn)
            for sim_word, score in similars:
                if score > 0.5:  # فقط کلمات با شباهت بالا
                    similar_words.add(sim_word.lower())
        except KeyError:
            print(f"کلمه '{word}' در مدل یافت نشد")
    return similar_words

# کلمات اصلی
base_words = ['warranty', 'guarantee', 'Warranty', 'Guarantee']

# پیدا کردن کلمات مشابه
similar_words = get_similar_words(word2vec_model, base_words)

# اضافه کردن غلط‌های املایی رایج
typos = {
    'waranty', 'warenty', 'warrenty', 'warrantee', 'warrantie', 'warranties',
    'guarante', 'garantee', 'guarentee', 'gaurantee', 'guaranty', 'guarantees',
    'guarranty', 'warrranty'
}

# ترکیب همه کلمات
all_warranty_words = similar_words.union(typos).union({'warranty', 'guarantee', 'warranties', 'guarantees'})

print(f"تعداد کل کلمات مرتبط با ضمانت: {len(all_warranty_words)}")
print("\nکلمات:")
print(sorted(all_warranty_words))

In [ ]:
# تابع بررسی وجود کلمات ضمانت در متن
def contains_warranty_words(text, warranty_words):
    if pd.isna(text):
        return False
    text_lower = str(text).lower()
    for word in warranty_words:
        if word.lower() in text_lower:
            return True
    return False

# فیلتر کردن نظرات مرتبط با ضمانت
train_df['has_warranty_mention'] = train_df['reviewText'].apply(
    lambda x: contains_warranty_words(x, all_warranty_words)
)

warranty_reviews = train_df[train_df['has_warranty_mention']]

print(f"تعداد کل نظرات: {len(train_df):,}")
print(f"تعداد نظرات مرتبط با ضمانت: {len(warranty_reviews):,}")
print(f"درصد: {len(warranty_reviews)/len(train_df)*100:.2f}%")

In [ ]:
# محاسبه میانگین امتیاز هر محصول برای نظرات مرتبط با ضمانت
warranty_product_ratings = warranty_reviews.groupby('asin').agg({
    'overall': ['mean', 'count']
}).reset_index()
warranty_product_ratings.columns = ['asin', 'warranty_avg_rating', 'warranty_review_count']

# ادغام با اطلاعات محصول
warranty_product_ratings = warranty_product_ratings.merge(title_brand_df, on='asin', how='left')

# فیلتر محصولاتی که حداقل ۵ نظر مرتبط با ضمانت دارند
warranty_product_ratings = warranty_product_ratings[warranty_product_ratings['warranty_review_count'] >= 5]

# مرتب‌سازی
warranty_product_ratings = warranty_product_ratings.sort_values('warranty_avg_rating', ascending=False)

print("📋 میانگین امتیاز رضایت از ضمانت کالاها:")
print("=" * 80)
print(f"{'برند':<25}{'میانگین امتیاز ضمانت':<20}{'تعداد نظرات':<15}")
print("-" * 80)

# نمایش ۲۰ محصول برتر
for idx, row in warranty_product_ratings.head(20).iterrows():
    brand = str(row['brand'])[:23] if pd.notna(row['brand']) else 'نامشخص'
    print(f"{brand:<25}{row['warranty_avg_rating']:<20.2f}{int(row['warranty_review_count']):<15}")

# ذخیره نتایج
warranty_product_ratings.to_csv('warranty_satisfaction_ratings.csv', index=False)
print("\n✅ نتایج در فایل 'warranty_satisfaction_ratings.csv' ذخیره شد.")

In [ ]:
# تحلیل کلی رضایت از ضمانت
print("\n📊 تحلیل کلی رضایت از ضمانت:")
print("=" * 50)
print(f"میانگین کلی امتیاز نظرات مرتبط با ضمانت: {warranty_reviews['overall'].mean():.2f}")
print(f"میانگین کلی امتیاز همه نظرات: {train_df['overall'].mean():.2f}")

# توزیع امتیازات نظرات مرتبط با ضمانت
plt.figure(figsize=(10, 5))
warranty_reviews['overall'].value_counts().sort_index().plot(kind='bar', color='coral', edgecolor='black')
plt.xlabel('امتیاز')
plt.ylabel('تعداد')
plt.title('توزیع امتیازات در نظرات مرتبط با ضمانت')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

In [ ]:
# نصب کتابخانه‌های مورد نیاز
!pip install transformers datasets accelerate -q
!pip install scikit-learn -q

import torch
from torch.utils.data import Dataset, DataLoader
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback
)
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, classification_report, confusion_matrix
import numpy as np

In [ ]:
# پیش‌پردازش متن
def preprocess_text(text, summary=None):
    """پیش‌پردازش متن نظر"""
    if pd.isna(text):
        text = ""
    else:
        text = str(text)

    # اضافه کردن خلاصه در صورت وجود
    if summary and not pd.isna(summary):
        text = f"{summary}. {text}"

    # پاکسازی
    text = re.sub(r'http\S+', '', text)  # حذف لینک‌ها
    text = re.sub(r'<[^>]+>', '', text)  # حذف تگ‌های HTML
    text = re.sub(r'\s+', ' ', text).strip()  # نرمال‌سازی فاصله‌ها

    return text

# آماده‌سازی داده‌ها
train_df['processed_text'] = train_df.apply(
    lambda row: preprocess_text(row['reviewText'], row['summary']), axis=1
)

# فیلتر کردن نظرات خیلی کوتاه یا خالی
train_df = train_df[train_df['processed_text'].str.len() > 10]

# تبدیل برچسب‌ها (1-5 به 0-4)
train_df['label'] = train_df['overall'] - 1

print(f"تعداد نمونه‌های آموزش پس از پیش‌پردازش: {len(train_df):,}")

In [ ]:
# نمونه‌گیری برای کاهش حجم (اختیاری - برای محدودیت منابع)
SAMPLE_SIZE = 150000  # می‌توانید افزایش دهید

if len(train_df) > SAMPLE_SIZE:
    # نمونه‌گیری طبقه‌بندی‌شده
    train_sample = train_df.groupby('label', group_keys=False).apply(
        lambda x: x.sample(min(len(x), SAMPLE_SIZE // 5), random_state=42)
    )
else:
    train_sample = train_df

print(f"تعداد نمونه‌های انتخاب‌شده: {len(train_sample):,}")
print("\nتوزیع برچسب‌ها:")
print(train_sample['label'].value_counts().sort_index())

In [ ]:
# تقسیم به آموزش و اعتبارسنجی
X_train, X_val, y_train, y_val = train_test_split(
    train_sample['processed_text'].values,
    train_sample['label'].values,
    test_size=0.15,
    random_state=42,
    stratify=train_sample['label'].values
)

print(f"تعداد نمونه آموزش: {len(X_train):,}")
print(f"تعداد نمونه اعتبارسنجی: {len(X_val):,}")

In [ ]:
# انتخاب مدل پایه
MODEL_NAME = "roberta-base"  # سبک‌تر و سریع‌تر
# یا می‌توانید استفاده کنید: "bert-base-uncased", "roberta-base"

# بارگذاری توکنایزر
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# تعریف Dataset
class SentimentDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_length=256):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = str(self.texts[idx])
        label = self.labels[idx]

        encoding = self.tokenizer(
            text,
            truncation=True,
            padding='max_length',
            max_length=self.max_length,
            return_tensors='pt'
        )

        return {
            'input_ids': encoding['input_ids'].flatten(),
            'attention_mask': encoding['attention_mask'].flatten(),
            'labels': torch.tensor(label, dtype=torch.long)
        }

# ایجاد دیتاست‌ها
train_dataset = SentimentDataset(X_train, y_train, tokenizer)
val_dataset = SentimentDataset(X_val, y_val, tokenizer)

In [ ]:
# محاسبه وزن کلاس‌ها برای مقابله با عدم توازن
from sklearn.utils.class_weight import compute_class_weight

class_weights = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(y_train),
    y=y_train
)
class_weights = torch.tensor(class_weights, dtype=torch.float)
print("وزن کلاس‌ها:", class_weights)

In [ ]:
# بارگذاری مدل
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=5,
    problem_type="single_label_classification"
)

# انتقال به GPU در صورت وجود
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")
model = model.to(device)
class_weights = class_weights.to(device)

In [ ]:
# تابع محاسبه متریک‌ها
def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    predictions = np.argmax(predictions, axis=1)

    f1_micro = f1_score(labels, predictions, average='micro')
    f1_macro = f1_score(labels, predictions, average='macro')
    f1_weighted = f1_score(labels, predictions, average='weighted')

    return {
        'f1_micro': f1_micro,
        'f1_macro': f1_macro,
        'f1_weighted': f1_weighted
    }

# تنظیمات آموزش
training_args = TrainingArguments(
    output_dir='./sentiment_model',
    num_train_epochs=4,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=64,
    warmup_steps=500,
    weight_decay=0.01,
    logging_dir='./logs',
    logging_steps=100,
    eval_strategy="steps",
    eval_steps=500,
    save_strategy="steps",
    save_steps=500,
    load_best_model_at_end=True,
    metric_for_best_model='f1_micro',
    greater_is_better=True,
     learning_rate=2e-5,
    warmup_ratio=0.1,
    fp16=torch.cuda.is_available(),  # Mixed precision برای سرعت بیشتر
    dataloader_num_workers=2,
)

# تعریف Custom Trainer با وزن کلاس‌ها
class WeightedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits = outputs.logits

        loss_fn = torch.nn.CrossEntropyLoss(weight=class_weights)
        loss = loss_fn(logits, labels)

        return (loss, outputs) if return_outputs else loss

# ایجاد Trainer
trainer = WeightedTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)]
)

In [ ]:
from torch.nn import CrossEntropyLoss
loss_fn = CrossEntropyLoss(label_smoothing=0.1)

In [ ]:
# آموزش مدل
print("🚀 شروع آموزش مدل...")
trainer.train()
print("✅ آموزش مدل به پایان رسید!")

In [ ]:
# ارزیابی روی داده‌های اعتبارسنجی
print("\n📊 ارزیابی مدل روی داده‌های اعتبارسنجی:")
eval_results = trainer.evaluate()
print(eval_results)

# پیش‌بینی روی داده‌های اعتبارسنجی
predictions = trainer.predict(val_dataset)
pred_labels = np.argmax(predictions.predictions, axis=1)

# گزارش طبقه‌بندی
print("\n📋 گزارش طبقه‌بندی:")
print(classification_report(y_val, pred_labels, target_names=['1', '2', '3', '4', '5']))

# ماتریس درهم‌ریختگی
plt.figure(figsize=(8, 6))
cm = confusion_matrix(y_val, pred_labels)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['1', '2', '3', '4', '5'],
            yticklabels=['1', '2', '3', '4', '5'])
plt.xlabel('پیش‌بینی‌شده')
plt.ylabel('واقعی')
plt.title('ماتریس درهم‌ریختگی')
plt.tight_layout()
plt.show()

In [ ]:
# پیش‌پردازش داده‌های آزمون
test_df['processed_text'] = test_df.apply(
    lambda row: preprocess_text(row['reviewText'], row['summary']), axis=1
)

# ایجاد دیتاست آزمون
class TestDataset(Dataset):
    def __init__(self, texts, tokenizer, max_length=256):
        self.texts = texts
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = str(self.texts[idx])

        encoding = self.tokenizer(
            text,
            truncation=True,
            padding='max_length',
            max_length=self.max_length,
            return_tensors='pt'
        )

        return {
            'input_ids': encoding['input_ids'].flatten(),
            'attention_mask': encoding['attention_mask'].flatten(),
        }

test_dataset = TestDataset(test_df['processed_text'].values, tokenizer)

In [ ]:
# پیش‌بینی
print("🔮 در حال پیش‌بینی روی داده‌های آزمون...")
test_predictions = trainer.predict(test_dataset)
test_pred_labels = np.argmax(test_predictions.predictions, axis=1)

# تبدیل به برچسب اصلی (1-5)
test_pred_labels = test_pred_labels + 1

print(f"✅ پیش‌بینی {len(test_pred_labels):,} نمونه انجام شد!")
print("\nتوزیع پیش‌بینی‌ها:")
print(pd.Series(test_pred_labels).value_counts().sort_index())